# Week 5 Presentation Brief — Variation B
## 🐟 Aquaculture Profit: Optimal Stocking Density for Barramundi
**SCIE1500**

> Work through all parts during the Week 5 lab. Your **10-minute Week 6 presentation** should cover: the problem, your model, optimal stocking density, and disease risk
> **What to submit:** every group member must individually upload their own copy of the same completed presentation slides (PDF or PowerPoint) to the LMS after your presentation — this lets your instructor verify who participated.


---
## 📋 Scenario

![Aquaculture profit as a function of stocking density](../images/W5B_aquaculture.svg)

A barramundi farm in Exmouth currently operates at a stocking density of **40 fish per 100 m³** and wants to know whether this is optimal. Profit (thousands of dollars/year) as a function of stocking density $d$ (fish per 100 m³):

$$P(d) = -0.5d^2 + 55d - 800$$

**Constraints:** $10 \leq d \leq 80$ fish/100 m³

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Optimise stocking density using calculus | ~25 min |
| B | Find break-even density and interpret marginal profit | ~20 min |
| C | Compare profit outcomes under disease and aeration scenarios | ~15 min |
| D | Verify the critical point symbolically using SymPy | ~10 min |

In [ ]:
# Run first — loads libraries for this session
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Barramundi profit model
def P(d):
    'Annual profit (thousands $) at stocking density d fish/100m³.'
    return -0.5*d**2 + 55*d - 800

def dP(d):
    'Marginal profit ($/fish per 100m³).'
    return -d + 55

d_min, d_max = 10, 80    # operational constraints

print("P(10) =", P(10), "($000)")
print("P(55) =", P(55), "($000)")
print("P(80) =", P(80), "($000)")

---
## Part A: Optimization (~25 min)

In [ ]:
# A.1 — Critical point: P'(d) = -d + 55 = 0  →  d* = 55
d_crit = 55
P_crit = P(d_crit)
print(f"P'(d) = -d + 55 = 0  →  d* = {d_crit} fish/100m³")
print(f"P''(d) = -1 < 0  →  maximum confirmed")
print(f"P({d_crit}) = ${P_crit:,.0f} thousand/year")

# Feasibility check
feasible = d_min <= d_crit <= d_max
print(f"Feasible (within [{d_min}, {d_max}])? {feasible}")

Compare critical point to boundaries.

In [ ]:
# A.2 — Compare critical point to boundaries
for d_s, label in [(d_crit, "Optimal (d=55)"), (d_min, "Min density (d=10)"), (d_max, "Max density (d=80)")]:
    print(f"{label}: P = ${P(d_s):,.0f}k  feasible? {d_min <= d_s <= d_max}")

Plot profit curve.

In [ ]:
# A.3 — Plot profit curve
d_vals = np.linspace(5, 90, 300)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(d_vals, P(d_vals), "steelblue", lw=2, label="Profit P(d)")
ax.plot(d_crit, P_crit, "ro", ms=12, label=f"Optimal: d={d_crit}, P=${P_crit:.0f}k")
ax.axvline(d_min, color="orange", ls="--", alpha=0.7, label=f"Min density ({d_min})")
ax.axvline(d_max, color="orange", ls="--", alpha=0.7, label=f"Max density ({d_max})")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Stocking Density d (fish per 100 m³)")
ax.set_ylabel("Annual Profit ($000)")
ax.set_title("Barramundi Aquaculture: Stocking Density Optimization")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Farm decision checkpoint

Use the graph to compare the farm’s current practice with the model’s recommendation.

- At the current density of 40 fish per 100 m³, use the sign and units of $P'(40)$ to explain what a small density increase predicts.
- Explain why the feasible maximum at 55 fish per 100 m³ is not an instruction to ignore welfare or disease risk.
- State one observation the farm should monitor if it chooses to move toward the calculated optimum.

---
## Part B: Break-Even and Marginal Profit (~20 min)

In [ ]:
# B.1 — Break-even densities: solve P(d) = 0
# -0.5d² + 55d - 800 = 0  →  d² - 110d + 1600 = 0
a, b, c = -0.5, 55, -800
disc = b**2 - 4*a*c
d_be1 = (-b - np.sqrt(disc)) / (2*a)
d_be2 = (-b + np.sqrt(disc)) / (2*a)
if d_be1 > d_be2: d_be1, d_be2 = d_be2, d_be1

print(f"Break-even densities: d = {d_be1:.1f} and d = {d_be2:.1f} fish/100m³")
print(f"Farm is profitable for d ∈ [{d_be1:.1f}, {d_be2:.1f}]")

# Marginal profit at the farm's current operating density (see Scenario)
d_current = 40
mp = dP(d_current)
print(f"\nMarginal profit at d = {d_current}: P'({d_current}) = {mp} ($k / fish per 100m³)")
print(f"→ Increasing density by 1 {'increases' if mp > 0 else 'decreases'} profit by ${abs(mp):.0f}k")

---
## Part C: Scenario Analysis (~15 min)

Two risks/opportunities could change the farm's economics: a **disease outbreak** would add a large fixed cost (biosecurity, treatment, stock losses) without changing how density itself affects yield, while an **aeration upgrade** would change the underlying yield relationship — both the marginal benefit and the crowding penalty of higher density. We model disease as the same profit curve with a bigger constant subtracted (\$1,200k instead of \$800k — an extra \$400k in disease-related fixed costs), and aeration as a genuinely different quadratic ($-0.4d^2+60d-900$).

✏️ **Before running the cell below:** one of these two scenarios keeps the optimal density at $d^*=55$ (the same as baseline), and the other moves it. Which do you expect changes the optimal density, and which doesn't — and why? Think about what a pure constant shift does to a function's derivative, versus what changing the quadratic and linear coefficients does.

In [ ]:
# C.1 — Disease and aeration scenarios
def P_disease(d):
    'Profit under disease outbreak (-400k fixed costs).'
    return -0.5*d**2 + 55*d - 1200

def P_aeration(d):
    'Profit with aeration upgrade (improved yield).'
    return -0.4*d**2 + 60*d - 900

# Aeration changes the coefficients, so (unlike disease) its optimal density
# is genuinely different from the baseline -- see the question above.
d_aer_crit = 60 / 0.8    # P_aeration'(d) = -0.8d + 60 = 0
d_vals = np.linspace(5, 90, 300)

print("Optimal results by scenario:")
print(f"  Baseline:  d* = {d_crit:.0f}, P* = ${P_crit:.0f}k")
print(f"  Disease:   d* = {d_crit:.0f}, P* = ${P_disease(d_crit):.0f}k  (same d, lower profit)")
print(f"  Aeration:  d* = {d_aer_crit:.0f}, P* = ${P_aeration(d_aer_crit):.0f}k")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(d_vals, P(d_vals),         "steelblue", lw=2, label="Baseline")
ax.plot(d_vals, P_disease(d_vals), "firebrick", lw=2, ls="--", label="Disease outbreak")
ax.plot(d_vals, P_aeration(d_vals),"green",     lw=2, ls=":",  label="Aeration upgrade")
ax.axvline(d_min, color="orange", ls="--", alpha=0.5)
ax.axvline(d_max, color="orange", ls="--", alpha=0.5)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Stocking Density d (fish per 100 m³)")
ax.set_ylabel("Annual Profit ($000)")
ax.set_title("Barramundi: Scenario Comparison")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

✏️ **Farm Manager Advisory (4–5 sentences):**
- Recommended stocking density and expected profit
- Disease risk: does the farm still break even under the disease scenario?
- Aeration upgrade: is it worthwhile? (Estimate the cost it would need to pay off at)

```
ADVISORY:
...
```

---
## Part D: Symbolic Differentiation with SymPy (~10 min)

In Part A you found $P'(d)$ **by hand** using the power rule, then solved $P'(d)=0$ for the critical point. Python's **SymPy** library performs this kind of algebra automatically — the same **symbolic computation** idea you used on paper — and it's the standard tool for calculus in Python throughout this unit.

Run the cell below to have SymPy differentiate $P(d)$ symbolically and solve for the critical point, and confirm it matches your by-hand result from Part A.

In [ ]:
# D.1 — Symbolic differentiation with SymPy
import sympy as sp

d_sym = sp.symbols('d')
P_expr = -0.5*d_sym**2 + 55*d_sym - 800

dP_expr = sp.diff(P_expr, d_sym)
print(f"P(d)  = {P_expr}")
print(f"P'(d) = {dP_expr}   ← same as the by-hand result from Part A")

# Solve P'(d) = 0 symbolically — same critical point as Part A.1
critical_points = sp.solve(sp.Eq(dP_expr, 0), d_sym)
print(f"\nCritical point(s): d = {critical_points}")

**Why this matters:** `sp.diff()` performs **symbolic differentiation** — it manipulates the algebraic expression using the same rules (power rule, etc.) you use on paper, rather than approximating a derivative from numbers the way `numpy` would. `sp.solve()` then finds the exact critical point algebraically.

---
## ✅ Presentation Checklist (Week 6, 10 minutes)

1. **Problem** (~2 min): Explain the trade-off between stocking density, yield, and mortality.
2. **Model** (~3 min): Present the profit function, take the derivative, and find the critical point.
3. **Results** (~3 min): Report optimal density, break-even range, and scenario comparison.
4. **Recommendation** (~2 min): Recommend a stocking density and propose risk mitigation steps.

---
## 📊 Presentation Marking Rubric (20 marks → scaled to 4% of your unit grade)

Your group presentation is graded out of 20 marks (scaled to 4% of your unit grade — each group presents twice, for 8% total). This rubric determines your group mark, which will be the mark for each contributing member unless we are advised otherwise.

| Criterion | Excellent | Good | Developing | Poor |
|---|---|---|---|---|
| **Problem Formulation** (5 marks) | Clear explanation of the real-world problem; audience understands what question is being answered and why it matters (5) | Problem explained but lacks full context or motivation (3–4) | Problem stated but unclear why it's important (1–2) | No clear problem statement (0) |
| **Mathematical Approach** (5 marks) | Correct model/method selected; clear justification for the choice; key equations presented clearly (5) | Correct approach with minor errors; justification present but weak (3–4) | Approach has errors or is poorly justified (1–2) | Wrong method or no mathematical content shown (0) |
| **Results & Interpretation** (6 marks) | Results are correct and clearly presented; findings are connected to a real-world decision, including limitations/trade-offs (6) | Results mostly correct; some interpretation but lacks depth (4–5) | Results unclear, minor errors, or interpretation is minimal — just states numbers (2–3) | Major errors, no results shown, or no interpretation given (0–1) |
| **Communication Quality** (4 marks) | Effective graphs/visuals support the story; all members participate and speak without reading from notes; well-rehearsed and within the 10-minute limit (4) | Visuals adequate; most members participate; slightly over/under time (3) | Visualization ineffective or missing; uneven participation; timing issues (1–2) | No visuals; one person dominates; major timing problems (0) |

**Total: ____ / 20 marks**